# 🧪 Outil de Prédiction de Cytotoxicité pour les CARs (Version Corrigée)

Ce notebook utilise le modèle XGBoost optimisé pour prédire si une nouvelle construction de CAR aura une cytotoxicité 'Haute' ou 'Basse' à partir de ses séquences protéiques.

### Instructions
1.  **Exécutez la Cellule 1** pour installer toutes les bibliothèques nécessaires.
2.  **Exécutez la Cellule 2** et utilisez le bouton "Parcourir" pour importer les trois fichiers sauvegardés : `final_xgb_model.joblib`, `data_scaler.joblib`, et `model_columns.joblib`.
3.  **Exécutez la Cellule 3** pour charger les modèles et les fonctions. *Cette étape peut prendre quelques minutes lors du premier chargement du modèle ESM-2.*
4.  **Choisissez une méthode de prédiction :**
    *   **Prédiction Unique :** Remplissez les champs de la **Cellule 4** et exécutez les **Cellules 5 et 6**.
    *   **Prédiction par Lot (Excel) :** Exécutez directement la **Cellule 7**.

### Étape 1 : Installation des Dépendances

In [1]:
# ==============================================================================
# Cellule 1 (Corrigée pour Inférence)
# ==============================================================================
# Ajout de 'xlsxwriter' à la liste des bibliothèques à installer
%pip install joblib scikit-learn pandas numpy torch transformers biopython xgboost tqdm xlsxwriter -q

print("✅ Bibliothèques nécessaires, y compris le moteur Excel (xlsxwriter), installées.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 7.6 MB/s eta 0:00:00
✅ Bibliothèques nécessaires, y compris le moteur Excel (xlsxwriter), installées.


### Étape 2 : Importation des Fichiers du Modèle

Exécutez la cellule ci-dessous et importez les 3 fichiers `.joblib` lorsque vous y êtes invité.

In [2]:
# ==============================================================================
# Cellule 2 (Version Complète et Corrigée)
# ==============================================================================
from google.colab import files
import os

print("Veuillez importer les 3 fichiers requis : 'final_xgb_model.joblib', 'data_scaler.joblib', et 'model_columns.joblib'")
uploaded = files.upload()

for fn in uploaded.keys():
  print(f'Fichier "{fn}" importé avec succès.')

# Mise à jour de la liste des fichiers requis pour le pipeline SANS PCA
required_files = [
    'final_xgb_model.joblib',
    'data_scaler.joblib',
    'model_columns.joblib'
]

all_files_present = all(os.path.exists(f) for f in required_files)

if all_files_present:
    print("\n✅ Tous les fichiers requis sont présents.")
else:
    print("\n❌ ERREUR : Un ou plusieurs fichiers requis sont manquants. Veuillez réessayer.")
    missing = [f for f in required_files if not os.path.exists(f)]
    print(f"Fichiers manquants : {missing}")

Veuillez importer les 3 fichiers requis : 'final_xgb_model.joblib', 'data_scaler.joblib', et 'model_columns.joblib'


Saving model_columns.joblib to model_columns.joblib
Saving final_xgb_model.joblib to final_xgb_model.joblib
Saving data_scaler.joblib to data_scaler.joblib
Fichier "model_columns.joblib" importé avec succès.
Fichier "final_xgb_model.joblib" importé avec succès.
Fichier "data_scaler.joblib" importé avec succès.

✅ Tous les fichiers requis sont présents.


### Étape 3 : Chargement des Modèles et Définition des Fonctions

In [6]:
# ==============================================================================
# Cellule 3 (Version Complète et Corrigée) : Chargement des Modèles et Fonctions
# ==============================================================================
import pandas as pd
import numpy as np
import joblib
import torch
from transformers import EsmTokenizer, EsmModel
from Bio.SeqUtils.ProtParam import ProteinAnalysis
import warnings

warnings.filterwarnings('ignore')

# --- Fonctions de Pré-traitement ---

def validate_and_clean_sequence(sequence, domain_name):
    """
    Valide qu'une séquence ne contient que des caractères d'acides aminés valides et la nettoie.
    Retourne la séquence nettoyée et une liste d'erreurs (vide si valide).
    """
    valid_aas = "ACDEFGHIKLMNPQRSTVWY*"
    errors = []

    if not isinstance(sequence, str) or not sequence:
        return "", errors # Retourne une chaîne vide pour les entrées non valides

    cleaned_seq = str(sequence).upper().replace(' ', '').strip()

    for i, char in enumerate(cleaned_seq):
        if char not in valid_aas:
            context_start = max(0, i - 10)
            context_end = min(len(cleaned_seq), i + 11)
            context = cleaned_seq[context_start:i] + f"->{char}<-" + cleaned_seq[i+1:context_end]
            error_msg = f"Domaine '{domain_name}': Caractère invalide '{char}' trouvé à la position {i}. Contexte: ...{context}..."
            errors.append(error_msg)

    return cleaned_seq, errors

def calculate_prot_features(sequence):
    """Calcule les caractéristiques biophysiques."""
    default_features = {'length': 0, 'mol_weight': 0, 'pI': 0, 'aromaticity': 0, 'instability_idx': 0, 'gravy': 0}
    if pd.isna(sequence) or not sequence: return default_features
    sequence_cleaned = str(sequence).upper().strip().rstrip('*')
    if not sequence_cleaned:
        default_features['length'] = len(str(sequence))
        return default_features
    try:
        analysis = ProteinAnalysis(sequence_cleaned)
        return {
            'length': len(str(sequence)),
            'mol_weight': analysis.molecular_weight(),
            'pI': analysis.isoelectric_point(),
            'aromaticity': analysis.aromaticity(),
            'instability_idx': analysis.instability_index(),
            'gravy': analysis.gravy()
        }
    except Exception:
        return default_features

def get_single_embedding(sequence, model, tokenizer, device):
    """Génère l'embedding pour une seule séquence."""
    if not isinstance(sequence, str) or len(sequence) == 0:
        return np.zeros(model.config.hidden_size)
    with torch.no_grad():
        inputs = tokenizer(sequence, return_tensors="pt", padding=True, truncation=True, max_length=1022).to(device)
        outputs = model(**inputs)
        embedding = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
        return embedding

# --- Chargement des Modèles ---
print("Chargement des objets sauvegardés...")
final_model = joblib.load('final_xgb_model.joblib')
scaler = joblib.load('data_scaler.joblib')
model_columns = joblib.load('model_columns.joblib')

print("Chargement du modèle de langage de protéines ESM-2 (cela peut prendre quelques minutes)...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
esm_model_name = "facebook/esm2_t33_650M_UR50D"
tokenizer = EsmTokenizer.from_pretrained(esm_model_name)
esm_model = EsmModel.from_pretrained(esm_model_name).to(device)
esm_model.eval()

print(f"\n✅ Prêt à faire des prédictions sur le device : {device}")

Chargement des objets sauvegardés...
Chargement du modèle de langage de protéines ESM-2 (cela peut prendre quelques minutes)...


tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.61G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/566 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



✅ Prêt à faire des prédictions sur le device : cuda


---
### **Option A : Prédiction Unique**
---

In [ ]:
# ==============================================================================
# Cellule 4a : Séquences des Domaines du CAR
# ==============================================================================
#@title Sous-option 4a : Séquences des Domaines du CAR
#@markdown Choisissez un exemple ou entrez vos propres séquences ci-dessous.
example_to_load = 'Test Set Example (JAG1-F1)' #@param ["Validation Set Example (KM666_3XG4S)", "Test Set Example (JAG1-F1)"]

# Dictionnaire des séquences exemples
validation_example = {
    'peptide_signal_seq': 'METDTLLLWVLLLWPGSTG',
    'scfv_seq': 'QVQLQESGPGLVKPSQTLSICTVSGFSLASYNIIHWVRQPPGKLEWLGVIWAGGSTNYNSALMSRLTSIKDNSKNQVFLKMSSLTAADTAVYYCAKRSDDYSWFAYWGQGTLVTVSSGGGGSGGGGSGGGGSENQMTQSPSSLASVSGDRVTMTCRASSSVSSSYLHWYQQKSGKAPKWWIYSTSNLASGVPSRFGSGSGTDFTLTISSLQPEDFATTYCQQYSGYPITFGQGTKVEIKR',
    'hinge_seq': 'AEPKSPDKTHTCPPCPKDPK',
    'tm_seq': 'FWVLVVGGVLACYSLLVTVAFIIFWV',
    'tail_seq': 'RSKRSRLLHSDYMNMTPRRPGPTRKHYQPYAPPRDFAAYRSRDQRLPPDAHKPPGGGSFRTPIQEEQADAHSTLAKIRVKFSRSADAPAYQGGQNQLYNELNLGRREEYDVLDKRRGRDPEMGGKPRRKNPQEGLYNELQKDKMAEAYSEIGMKGERRRGKGHDCLYQGLSTATKDTYDALHMQALPPR'
}

test_example = {
    'peptide_signal_seq': 'MLLLVTSLLLCELPHPAFLLIP',
    'scfv_seq': 'DIQMTQSPSSLSASVGDRVTITCRASQSISSYLNWYQQKPGKAPKLLIYAASALQSGVPSRFSGSGSGTDFTLTISSLQPEDFATYYCQQAYYDPTTFGQGTKVEIKGSTSGSGKPGSGEGSTKGEVQLLESGGGLVQPGGSLRLSCAASGFTFSSYAMSWVRQAPGKGLEWVSTISTSGDYTTYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAKSTAYFDYWGQGTLVTVSS',
    'hinge_seq': 'AAATTTPAPRPPTPAPTIASQPLSLRPEACRPAAGGAVHTRGLDFACD',
    'tm_seq': 'FWVLVVVGGVLACYSLLVTVAFIIFWV',
    'tail_seq': 'RSKRSRLLHSDYMNMTPRRPGPTRKHYQPYAPPRDFAAYRSRKRGRKKLLYIFKQPFMRPVQTTQEEDGCSCRFPEEEEGGCELRVKFSRSADAPAYQQGQNQLYNELNLGRREEYDVLDKRRGRDPEMGGKPRRKNPQEGLYNELQKDKMAEAYSEIGMKGERRRGKGHDGLYQGLSTATKDTYDALHMQALPPR'
}

if 'Validation' in example_to_load:
    loaded_seqs = validation_example
else:
    loaded_seqs = test_example

peptide_signal_seq = loaded_seqs['peptide_signal_seq']
scfv_seq = loaded_seqs['scfv_seq']
hinge_seq = loaded_seqs['hinge_seq']
tm_seq = loaded_seqs['tm_seq']
tail_seq = loaded_seqs['tail_seq']

print(f"Exemple '{example_to_load}' chargé. Modifiez les champs ci-dessous si vous le souhaitez, puis exécutez la cellule.")

Exemple 'Test Set Example (JAG1-F1)' chargé. Modifiez les champs ci-dessous si vous le souhaitez, puis exécutez la cellule.


In [ ]:
# ==============================================================================
# Cellule 4b : Saisie Manuelle des Séquences
# ==============================================================================
#@title Sous-option 4b : Entrez les Séquences du CAR à Prédire Manuellement
#@markdown Remplissez les champs ci-dessous avec les séquences de chaque domaine, puis exécutez cette cellule pour les charger en mémoire.

peptide_signal_seq = "MLLLVTSLLLCELPHPAFLLIP" #@param {type:"string"}
scfv_seq = "DIQMTQSPSSLSASVGDRVTITCRASQSISSYLNWYQQKPGKAPKLLIYAASALQSGVPSRFSGSGSGTDFTLTISSLQPEDFATYYCQQAYYDPTTFGQGTKVEIKGSTSGSGKPGSGEGSTKGEVQLLESGGGLVQPGGSLRLSCAASGFTFSSYAMSWVRQAPGKGLEWVSTISTSGDYTTYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAKSTAYFDYWGQGTLVTVSS" #@param {type:"string"}
hinge_seq = "AAATTTPAPRPPTPAPTIASQPLSLRPEACRPAAGGAVHTRGLDFACD" #@param {type:"string"}
tm_seq = "FWVLVVVGGVLACYSLLVTVAFIIFWV" #@param {type:"string"}
tail_seq = "RSKRSRLLHSDYMNMTPRRPGPTRKHYQPYAPPRDFAAYRSRKRGRKKLLYIFKQPFMRPVQTTQEEDGCSCRFPEEEEGGCELRVKFSRSADAPAYQQGQNQLYNELNLGRREEYDVLDKRRGRDPEMGGKPRRKNPQEGLYNELQKDKMAEAYSEIGMKGERRRGKGHDGLYQGLSTATKDTYDALHMQALPPR" #@param {type:"string"}

print("✅ Séquences saisies et stockées dans les variables.")
print("Vous pouvez maintenant exécuter la 'Cellule 5 : Exécuter le Pipeline de Prédiction' pour obtenir le résultat.")

✅ Séquences saisies et stockées dans les variables.
Vous pouvez maintenant exécuter la 'Cellule 5 : Exécuter le Pipeline de Prédiction' pour obtenir le résultat.


### Étape 5 : Exécuter le Pipeline de Prédiction

In [ ]:
# ==============================================================================
# Cellule 5 (Complète et Corrigée) : Exécuter le Pipeline de Prédiction Unique
# ==============================================================================
print("Lancement du pipeline de prédiction...")

# 1. Collecter les séquences dans un dictionnaire
domain_inputs = {
    'Peptide_Signal': peptide_signal_seq,
    'scFv': scfv_seq,
    'Hinge': hinge_seq,
    'TM': tm_seq,
    'Tail': tail_seq
}

# --- NOUVEAU BLOC DE VALIDATION ---
print("Validation des séquences d'entrée...")
all_validation_errors = []
cleaned_domain_inputs = {}
for domain_name, sequence in domain_inputs.items():
    cleaned_seq, errors = validate_and_clean_sequence(sequence, domain_name)
    if errors:
        all_validation_errors.extend(errors)
    cleaned_domain_inputs[domain_name] = cleaned_seq

# Si des erreurs sont trouvées, les afficher et arrêter le pipeline
if all_validation_errors:
    print("\n❌ ERREUR : Des caractères invalides ont été détectés dans les séquences. Prédiction annulée.")
    for error in all_validation_errors:
        print(f"   - {error}")
    # Effacer tout résultat précédent pour éviter la confusion
    if 'single_prediction_result' in locals():
        del single_prediction_result
# Si tout est valide, continuer avec le pipeline de prédiction
else:
    print("✅ Séquences valides.")
    with np.errstate(all='ignore'): # Supprimer les warnings de calcul sur séquences vides
        # --- Début du Pipeline de Prédiction ---

        # Utiliser les séquences nettoyées
        domain_inputs = cleaned_domain_inputs

        features_dict = {}
        protein_domains_internal = ['Peptide_Signal', 'scFv', 'Hinge', 'TM', 'Tail']

        # 2. Calculer les caractéristiques biophysiques
        for domain in protein_domains_internal:
            biophys_feats = calculate_prot_features(domain_inputs[domain])
            for fname, fvalue in biophys_feats.items():
                features_dict[f"biophys_{domain}_{fname}"] = fvalue

        # 3. Calculer la caractéristique 'scFv_family'
        try:
            scfv_len = features_dict['biophys_scFv_length']
            if scfv_len <= 200: scfv_family = 'Short'
            elif scfv_len <= 300: scfv_family = 'Standard'
            else: scfv_family = 'Long'
        except KeyError:
            scfv_family = 'Standard' # Valeur par défaut

        features_dict['family_Short'] = 1 if scfv_family == 'Short' else 0
        features_dict['family_Standard'] = 1 if scfv_family == 'Standard' else 0
        features_dict['family_Long'] = 1 if scfv_family == 'Long' else 0

        # 4. Calculer les embeddings ESM-2
        print("Calcul des embeddings (peut prendre ~30 secondes)...")
        for domain in protein_domains_internal:
            embedding = get_single_embedding(domain_inputs[domain], esm_model, tokenizer, device)
            for i, val in enumerate(embedding):
                features_dict[f"emb_{domain}_{i}"] = val

        # 5. Créer le DataFrame
        X_new = pd.DataFrame([features_dict])

        # 6. Aligner les colonnes
        X_new_aligned = X_new.reindex(columns=model_columns, fill_value=0)

        # 7. Appliquer la mise à l'échelle
        print("Application de la mise à l'échelle...")
        X_new_scaled = scaler.transform(X_new_aligned)

        # SÉCURITÉ : Vérifier que le nombre final de colonnes correspond
        if X_new_aligned.shape[1] != final_model.n_features_in_:
             raise ValueError(f"Erreur de dimension ! Le modèle attend {final_model.n_features_in_} features, mais en a reçu {X_new_aligned.shape[1]}.")

        # 8. Prédiction
        prediction_proba = final_model.predict_proba(X_new_scaled)[0]
        prediction = final_model.predict(X_new_scaled)[0]

        print("\n--- Prédiction terminée ! ---")

        # Stocker les résultats pour la cellule d'affichage
        single_prediction_result = {'prediction': prediction, 'probability': prediction_proba[1]}

Lancement du pipeline de prédiction...


NameError: name 'peptide_signal_seq' is not defined

### Étape 6 : Résultats de la Prédiction Unique

In [ ]:
# ==============================================================================
# Cellule 6 (Complète et Corrigée) : Affichage Détaillé du Résultat
# ==============================================================================
from IPython.display import display, HTML

# Vérifier si un résultat de prédiction existe avant d'essayer de l'afficher
try:
    if 'single_prediction_result' in locals():
        confidence_score = single_prediction_result['probability']
        prediction = single_prediction_result['prediction']

        # --- NOUVEAU BLOC : PRÉPARATION DE L'AFFICHAGE DES SÉQUENCES ---

        # Fonction pour tronquer les longues séquences pour un affichage propre
        def truncate_sequence(seq, max_len=80):
            if len(seq) > max_len:
                return f"{seq[:35]}...{seq[-20:]}"
            return seq

        # Création du bloc HTML pour afficher les séquences utilisées
        sequences_html = f"""
        <div style="border: 1px solid #ccc; padding: 15px; border-radius: 8px; background-color: #f9f9f9; margin-bottom: 15px;">
            <h3 style="margin-top: 0;">Séquences utilisées pour cette prédiction :</h3>
            <ul style="font-family: monospace; list-style-type: none; padding-left: 0;">
                <li style="margin-bottom: 5px;"><strong>Peptide Signal:</strong> {truncate_sequence(peptide_signal_seq)}</li>
                <li style="margin-bottom: 5px;"><strong>scFv:</strong> {truncate_sequence(scfv_seq)}</li>
                <li style="margin-bottom: 5px;"><strong>Hinge:</strong> {truncate_sequence(hinge_seq)}</li>
                <li style="margin-bottom: 5px;"><strong>TM:</strong> {truncate_sequence(tm_seq)}</li>
                <li style="margin-bottom: 5px;"><strong>Tail:</strong> {truncate_sequence(tail_seq)}</li>
            </ul>
        </div>
        """

        # --- FIN DU NOUVEAU BLOC ---

        # Génération du bloc HTML pour le résultat de la prédiction
        if prediction == 1:
            result_html = f"""
            <div style="border: 2px solid green; padding: 20px; border-radius: 10px; background-color: #e8f5e9;">
                <h2 style="color: green; margin-top:0;">Prédiction : Haute Cytotoxicité</h2>
                <p style="font-size: 1.2em;">Le modèle est confiant à <strong>{confidence_score:.2%}</strong> que ce CAR sera efficace.</p>
            </div>
            """
        else:
            result_html = f"""
            <div style="border: 2px solid orange; padding: 20px; border-radius: 10px; background-color: #fff3e0;">
                <h2 style="color: orange; margin-top:0;">Prédiction : Basse Cytotoxicité</h2>
                <p style="font-size: 1.2em;">Le modèle est confiant à <strong>{1-confidence_score:.2%}</strong> que ce CAR sera peu efficace (score 'Haute' : {confidence_score:.2%}).</p>
            </div>
            """

        # Affichage combiné des séquences et du résultat
        display(HTML(sequences_html + result_html))

    else:
        # Gère le cas où la cellule 5 a été exécutée mais a échoué à la validation
        # (le message d'erreur est déjà affiché par la cellule 5)
        pass

except NameError:
    # Gère le cas où la cellule 5 n'a jamais été exécutée
    display(HTML("<div style='border: 1px solid #ccc; padding: 10px; border-radius: 5px; color: #555;'>Aucun résultat à afficher. Veuillez d'abord exécuter la Cellule 5 pour générer une prédiction.</div>"))

---
### **Option B : Prédiction par Lot (Fichier Excel)**
---

### Étape 7 : Prédire sur un Fichier Excel Complet

Si vous avez plusieurs constructs à prédire, exécutez la cellule ci-dessous pour importer un fichier Excel.

**Format requis :**
*   Le fichier doit être au format `.xlsx` ou `.xls`.
*   Les noms de colonnes pour les séquences **doivent être exactement** : `Peptide_Signal_(Protein)`, `scFv_(Protein)`, `Hinge_(Protein)`, `TM_(Protein)`, `Tail_(Protein)`.
*   Une colonne `Construct ID` est recommandée pour identifier vos séquences.

In [7]:
# ==============================================================================
# Cellule 7 (Version Complète et Corrigée) : Prédire sur un Fichier Excel
# ==============================================================================
from google.colab import files
from tqdm.auto import tqdm
import io

print("Veuillez importer votre fichier Excel contenant les séquences à prédire.")
uploaded_excel = files.upload()

if not uploaded_excel:
    print("Aucun fichier n'a été importé. Opération annulée.")
else:
    filename = next(iter(uploaded_excel))
    content = uploaded_excel[filename]

    try:
        df_to_predict = pd.read_excel(io.BytesIO(content))
        print(f"Fichier '{filename}' chargé avec succès. {len(df_to_predict)} constructs à prédire.")

        # --- Début du Pipeline de Prédiction en Batch ---
        all_rows_features = []
        validation_errors_list = []

        protein_domains_original_names = ['Peptide_Signal_(Protein)', 'AntigenBindingDomain_(Protein)', 'Hinge_(Protein)', 'TM_(Protein)', 'Tail_(Protein)']
        protein_domains_internal = ['Peptide_Signal', 'AntigenBindingDomain', 'Hinge', 'TM', 'Tail']
        rename_map = {orig: clean for orig, clean in zip(protein_domains_original_names, protein_domains_internal)}

        df_processed = df_to_predict.rename(columns=rename_map)

        progress_bar = tqdm(df_processed.iterrows(), total=len(df_processed), desc="Traitement des constructs")

        for index, row in progress_bar:
            row_features = {}
            row_errors = []

            # --- BLOC DE VALIDATION PAR LIGNE ---
            cleaned_row_sequences = {}
            for domain in protein_domains_internal:
                original_sequence = row.get(domain)
                cleaned_seq, errors = validate_and_clean_sequence(original_sequence, domain)
                if errors:
                    row_errors.extend(errors)
                cleaned_row_sequences[domain] = cleaned_seq

            validation_status = ' | '.join(row_errors) if row_errors else 'OK'
            validation_errors_list.append(validation_status)

            # Si la ligne a une erreur de séquence, on crée quand même des features "vides"
            # mais on ne les utilisera pas pour la prédiction finale de cette ligne
            if validation_status != 'OK':
                all_rows_features.append({}) # Dictionnaire vide comme marqueur
                continue
            # --- FIN DU BLOC DE VALIDATION ---

            with np.errstate(all='ignore'):
                # Calcul des caractéristiques biophysiques
                for domain in protein_domains_internal:
                    biophys_feats = calculate_prot_features(cleaned_row_sequences[domain])
                    for fname, fvalue in biophys_feats.items():
                        row_features[f"biophys_{domain}_{fname}"] = fvalue

                # Calcul de la caractéristique 'scFv_family'
                try:
                    scfv_len = row_features['biophys_AntigenBindingDomain_length']
                    if scfv_len <= 200: scfv_family = 'Short'
                    elif scfv_len <= 300: scfv_family = 'Standard'
                    else: scfv_family = 'Long'
                except KeyError:
                    scfv_family = 'Standard'

                row_features['family_Short'] = 1 if scfv_family == 'Short' else 0
                row_features['family_Standard'] = 1 if scfv_family == 'Standard' else 0
                row_features['family_Long'] = 1 if scfv_family == 'Long' else 0

                # Calcul des embeddings ESM-2
                for domain in protein_domains_internal:
                    embedding = get_single_embedding(cleaned_row_sequences[domain], esm_model, tokenizer, device)
                    for i, val in enumerate(embedding):
                        row_features[f"emb_{domain}_{i}"] = val

            all_rows_features.append(row_features)

        # --- Création du DataFrame final et prédictions ---
        # Filtrer les lignes avec des erreurs avant de créer le DataFrame pour le modèle
        valid_rows_indices = [i for i, status in enumerate(validation_errors_list) if status == 'OK']
        valid_rows_features = [all_rows_features[i] for i in valid_rows_indices]

        if valid_rows_features:
            X_new_batch = pd.DataFrame(valid_rows_features)
            X_new_aligned = X_new_batch.reindex(columns=model_columns, fill_value=0)

            print("\nApplication de la mise à l'échelle au lot de données valides...")
            X_new_scaled_batch = scaler.transform(X_new_aligned)

            final_predictions_proba = final_model.predict_proba(X_new_scaled_batch)
            final_predictions = final_model.predict(X_new_scaled_batch)

            confidences = final_predictions_proba[:, 1]
            predictions = ['High' if pred == 1 else 'Low' for pred in final_predictions]
        else:
            print("\nAucune ligne valide trouvée pour la prédiction.")
            predictions = []
            confidences = []

        # Ajouter les résultats au DataFrame original
        df_to_predict['Validation_Status'] = validation_errors_list
        # Initialiser les colonnes de prédiction avec des valeurs par défaut
        df_to_predict['Cytotoxicité_Prédite'] = 'Erreur de Validation'
        df_to_predict['Confiance_Prediction_High'] = np.nan
        # Remplir les prédictions uniquement pour les lignes valides
        df_to_predict.loc[valid_rows_indices, 'Cytotoxicité_Prédite'] = predictions
        df_to_predict.loc[valid_rows_indices, 'Confiance_Prediction_High'] = confidences

        # Exporter le résultat
        output_filename = filename.replace('.xlsx', '').replace('.xls', '') + '_predictions.xlsx'
        with pd.ExcelWriter(output_filename, engine='xlsxwriter') as writer:
            df_to_predict.to_excel(writer, index=False, sheet_name='Predictions')
            workbook  = writer.book
            worksheet = writer.sheets['Predictions']
            percent_format = workbook.add_format({'num_format': '0.00%'})
            conf_col_idx = df_to_predict.columns.get_loc('Confiance_Prediction_High')
            worksheet.set_column(conf_col_idx, conf_col_idx, 25, percent_format)

        print(f"\n✅ Prédictions terminées. Le fichier '{output_filename}' a été généré.")
        print("Il contient une colonne 'Validation_Status' indiquant les erreurs éventuelles.")

    except Exception as e:
        print(f"\n❌ Une erreur est survenue lors du traitement du fichier Excel : {e}")
        print("Veuillez vérifier que le format du fichier et les noms des colonnes sont corrects.")

Veuillez importer votre fichier Excel contenant les séquences à prédire.


Saving dataset_inference_validation.xlsx to dataset_inference_validation (2).xlsx
Fichier 'dataset_inference_validation (2).xlsx' chargé avec succès. 10 constructs à prédire.


Traitement des constructs:   0%|          | 0/10 [00:00<?, ?it/s]


Application de la mise à l'échelle au lot de données valides...

✅ Prédictions terminées. Le fichier 'dataset_inference_validation (2)_predictions.xlsx' a été généré.
Il contient une colonne 'Validation_Status' indiquant les erreurs éventuelles.


La Cellule 8 prendra le DataFrame de résultats df_to_predict (créé et rempli dans la Cellule 7) et le formatera pour un affichage stylisé, similaire à ce que fait la Cellule 27 du notebook d'entraînement.

Notez qu'ici, nous n'avons pas de "vraie valeur" ou de notion de "prédiction correcte", car ce sont de nouvelles données. Le tableau se concentrera donc sur l'affichage clair des prédictions et des scores de confiance.


### **Etape 8 :** Visualisation des Résultats de Prédiction par Lot

In [8]:
# ==============================================================================
# Cellule 8 (Version Complète et Corrigée) : Visualisation des Résultats par Lot
# ==============================================================================
from IPython.display import display, HTML

# Vérifier si le DataFrame de prédiction 'df_to_predict' a été généré par la cellule précédente
try:
    if 'df_to_predict' in locals() and not df_to_predict.empty:
        print("\n--- Tableau Récapitulatif des Prédictions sur le Fichier Excel ---")

        # --- NOUVEAU BLOC : FONCTION DE TRONCATURE ---
        def truncate_sequence_for_display(seq, max_len=60):
            """Tronque une séquence pour l'affichage dans le DataFrame."""
            if isinstance(seq, str) and len(seq) > max_len:
                return f"{seq[:25]}...{seq[-15:]}"
            return seq
        # --- FIN DU NOUVEAU BLOC ---

        # Sélectionner les colonnes pertinentes pour l'affichage
        display_cols = [
            'Construct ID',
            'Validation_Status',
            'Cytotoxicité_Prédite',
            'Confiance_Prediction_High'
        ]

        sequence_cols_to_display = ['scFv_(Protein)', 'Hinge_(Protein)']
        for col in sequence_cols_to_display:
            if col in df_to_predict.columns:
                display_cols.append(col)

        display_cols = [col for col in display_cols if col in df_to_predict.columns]

        # Créer une copie pour le formatage
        results_to_style = df_to_predict[display_cols].copy()

        # --- APPLICATION DE LA TRONCATURE SUR LES COLONNES DE SÉQUENCES ---
        for col in sequence_cols_to_display:
            if col in results_to_style.columns:
                results_to_style[col] = results_to_style[col].apply(truncate_sequence_for_display)
        # --- FIN DE L'APPLICATION ---

        # Appliquer un style conditionnel pour une meilleure lisibilité
        def style_predictions_and_errors(df):
            style = pd.DataFrame('', index=df.index, columns=df.columns)
            mask_error = df['Validation_Status'] != 'OK'
            style.loc[mask_error, :] = 'background-color: #f2f2f2; color: #888'

            mask_high = (df['Cytotoxicité_Prédite'] == 'High') & (~mask_error)
            mask_low = (df['Cytotoxicité_Prédite'] == 'Low') & (~mask_error)

            style.loc[mask_high, :] = 'background-color: #e8f5e9'
            style.loc[mask_low, :] = 'background-color: #fff3e0'
            return style

        # Appliquer le formatage et le style
        styled_df = results_to_style.style.format({
            'Confiance_Prediction_High': '{:.2%}'
        }, na_rep="-").apply(style_predictions_and_errors, axis=None)

        display(styled_df)

    else:
        print("Aucun résultat de prédiction par lot à afficher. Veuillez d'abord exécuter la Cellule 7 avec un fichier Excel.")

except NameError:
    print("Aucun résultat de prédiction par lot à afficher. Veuillez d'abord exécuter la Cellule 7 avec un fichier Excel.")


--- Tableau Récapitulatif des Prédictions sur le Fichier Excel ---


,Construct ID,Validation_Status,Cytotoxicité_Prédite,Confiance_Prediction_High,Hinge_(Protein)
0,sdHER2-5_KIRS2DAP12,OK,Low,49.61%,AIEQKLISEEDLAIGSNSSDPLLVSVTGNPSNSWPSPTEPSSKTGNPRHLH
1,F265_KIRS2DAP12,OK,High,50.17%,AIEQKLISEEDLAIGSNSSDPLLVSVTGNPSNSWPSPTEPSSKTGNPRHLH
2,scFv2_EH2_Myc_KIRS2,OK,High,50.78%,SNSSDPLLVSVTGNPSNSWPSPTEPSSKTGNPRHLH
3,scFv1_EH2_Myc_KIRS2,OK,High,50.78%,SNSSDPLLVSVTGNPSNSWPSPTEPSSKTGNPRHLH
4,M410_BBz,OK,Low,49.30%,VSGTIEVMYPPPYLDNEKSNGTIIHVKGKHLCPSPLFPGPSKP
5,14g2a_KIRS2_DNAM1,OK,Low,49.19%,AIGSNSSDPLLVSVTGNPSNSWPSPTEPSSKTGNPRHLH
6,pHE_B7H3_28z_mKate2,OK,High,50.17%,IEVMYPPPYLDNEKSNGTIIHVKGKHLCPSPLFPGPSKP
7,pHUS 14g2a-Myc-KIRS2-dIL2RB-YXXQ BSD,OK,Low,49.19%,AIGSNSSDPLLVSVTGNPSNSWPSPTEPSSKTGNPRHLH
8,pHRSIN_RWG8_BBz,OK,High,50.37%,TTTPAPRPPTPAPTIASQPLSLRPEACRPAAGGAVHTRGLDFACD
9,pHRSIN_cita-cel_BCMA_BBz,OK,Low,49.65%,TSTTTPAPRPPTPAPTIASQPLSLRPEACRPAAGGAVHTRGLDFACD
